**BERN02 Exercise: Regression**

Name: Yang Shann Wen

Date: 1 September 2026

In [30]:
import pandas as pd
import numpy as np

In [31]:
# Import and load dataset
df = pd.read_csv('/data/pollution_cleaneddata.csv')

In [32]:
# Extract the vector of observations of the predictor: % of families with income < $3000 (POOR)
x_data = df['POOR']

# Extract the vector of observations of the response variable: Total age-adjusted mortality rate per 100,000 (MORT)
y_data = df['MORT']

# Convert into numpy array for calculations
x_data = np.array(x_data)
y_data = np.array(y_data)

In [33]:
# Function for performing predictions with Local Regression
def loess(y, x, k, x0):
  """
  Parameters:
  y: The response variable observations
  x: The predictor observations
  k: The number of nearest neighboring points to include in each local regression
  x0: The target values to be predicted

  Outputs:
  pred: The list of predicted expected values
  se: The list of standard deviations of the expected values

  """

  # Empty list to store my predicted expected values and standard errors
  pred = []
  se = []

  # Run the function for every target value in x0
  for i in x0:

    # Calculate distances between every data point x and the target value
    distances = np.abs(x-i)

    # Pick the k nearest points
    nearest_index = np.argsort(distances)[:k]

    # Fit an unweighted OLS line for the local area
    # Extract the x and y values for these k nearest points
    x_neighbors = x[nearest_index]
    y_neighbors = y[nearest_index]

    # Calculate the average of these k nearest points
    x_mean = np.mean(x_neighbors)
    y_mean = np.mean(y_neighbors)

    # Calculate the slope (beta_1): simple regression formula
    numerator = np.sum((x_neighbors-x_mean) * (y_neighbors-y_mean))
    denominator = np.sum((x_neighbors - x_mean)**2)
    beta_1 = numerator / denominator

    # Calculate the intercept (beta_0)
    beta_0 = y_mean - (beta_1 * x_mean)

    # Plug in target value to make predictions
    y_predicted = beta_0 + (beta_1 * i)
    pred.append(float(np.round(y_predicted, 4)))

    # Calculate the variance of the error (sigma^2)
    # The variance is estimated without weighting, using the unweighted RSS divided by the degrees of freedom (k - 2)
    e = y_neighbors - beta_0 - (beta_1 * x_neighbors)
    variance = (np.sum(e**2)) / (k-2)

    # Calculate the variance of the expected value
    h = ((i - x_mean)**2) / denominator
    variance_expected_value = variance * ((1/k) + h)

    # Square root for standard deviation
    standard_error = np.sqrt(variance_expected_value)
    se.append(float(np.round(standard_error, 4)))

  return pred, se

In [34]:
# Set parameters for prediction
k_values = [5, 10, 15]
x0 = [10, 18, 25]

# Loop through each k value
for k in k_values:
    # Run the function
    pred, se = loess(y_data, x_data, k, x0)

    print(f"\nPredictions when k = {k}:")
    for x_val, p, s in zip(x0, pred, se):
        print(f"  When x0 = {x_val}, MORT = {p}. (Standard error = {s}).")


Predictions when k = 5:
  When x0 = 10, MORT = 870.831. (Standard error = 21.6091).
  When x0 = 18, MORT = 971.4977. (Standard error = 24.4342).
  When x0 = 25, MORT = 1024.8835. (Standard error = 28.6186).

Predictions when k = 10:
  When x0 = 10, MORT = 901.8106. (Standard error = 20.935).
  When x0 = 18, MORT = 957.2334. (Standard error = 16.5458).
  When x0 = 25, MORT = 1012.1633. (Standard error = 27.5934).

Predictions when k = 15:
  When x0 = 10, MORT = 902.9093. (Standard error = 18.0756).
  When x0 = 18, MORT = 959.1905. (Standard error = 20.6523).
  When x0 = 25, MORT = 1009.0309. (Standard error = 25.4904).


As seen in the results, there is a consistent positive association between MORT and POOR, since MORT increases as POOR rises from 10% to 25%.

A k-value of 10 or 15 might be a better choice here. When k = 5, the predicted MORT for 10% poverty dropped significantly to ~870, suggesting the model is too sensitive to local noise, as well as yielding the highest standard error. When k = 10 and k = 15, the prediction stabilizes around 901-902 and standard errors decrease, achieving a better balance in the bias–variance trade-off.